In [21]:
# -*- coding: utf-8 -*-
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.ticker import LogLocator, NullFormatter

# =========
# Chemins (3 expériences)
# =========
DATA_PATHS = {
    "baseline": Path(
        "../experiments/aestra_baseline_1000_spectra_colab/postprocessing/data/periodograms.npz"
    ),
    "proxies": Path(
        "../experiments/aestra_baseline_1000_spectra_activity_proxies_colab/postprocessing/data/periodograms.npz"
    ),
    # "restframe": Path(
    #     "../experiments/aestra_baseline_1000_spectra_encode_in_rest_frame_colab/postprocessing/data/periodograms.npz"
    # ),
}

OUTDIR = Path("./plots_compare_periodograms")
OUTDIR.mkdir(parents=True, exist_ok=True)

# =========
# Réglages
# =========
DPI = 300
FIGSIZE_SINGLE = (8.5, 5.0)
USE_COMMON_RANGE = True
ANNOTATE_TOP = False
TOP_K = 1
PEAK_EXCLUDE_EDGE = 0.0
SHOW_TITLES = True

# Labels & styles
LABELS = {
    "proxies": "Proxies Concaténés",
    "baseline": "Baseline",
    "restframe": "Encode Rest-Frame",
}
LINESTYLES = {
    "proxies": "-",
    "baseline": "--",
    "restframe": ":",
}
LINEWIDTH = 1.6

# Série(s) à comparer
SERIES = [
    ("v_correct", r"$v_{\mathrm{correct}}$"),
]


# =========
# Helpers
# =========
def clean_series(periods, power):
    p = np.asarray(periods, float)
    y = np.asarray(power, float)
    m = np.isfinite(p) & np.isfinite(y) & (p > 0)
    p, y = p[m], y[m]
    if p.size == 0:
        return p, y
    idx = np.argsort(p)
    return p[idx], y[idx]


def normalize(y):
    # Ici: pas de normalisation (on garde la puissance en valeur absolue)
    return y


def intersect_range_multi(series_dict):
    pmins, pmaxs = [], []
    for p, _ in series_dict.values():
        if p.size:
            pmins.append(np.min(p))
            pmaxs.append(np.max(p))
    if not pmins:
        return None, None
    return max(pmins), min(pmaxs)


def crop_to_range(p, y, pmin, pmax):
    m = (p >= pmin) & (p <= pmax)
    return p[m], y[m]


def find_top_peaks(p, y, k=1, exclude_edge_frac=0.0):
    if p.size < 3:
        return []
    i0 = int(exclude_edge_frac * (p.size - 1))
    i1 = (p.size - 1) - i0
    i0 = max(i0, 1)
    i1 = max(i1, i0 + 1)
    idx_region = np.arange(i0, i1)
    if idx_region.size == 0:
        return []
    y_region = y[idx_region]
    order = np.argsort(y_region)[::-1]
    return list(idx_region[order[:k]])


def plot_many(ax, series_dict, title=None):
    """
    series_dict: dict key -> (p, y) déjà nettoyés et (éventuellement) recadrés
    """
    # Traces
    for key, (p, y) in series_dict.items():
        if p.size == 0:
            continue
        ax.plot(
            p, y, lw=LINEWIDTH, ls=LINESTYLES.get(key, "-"), label=LABELS.get(key, key)
        )

    # Axes
    ax.set_xscale("log")
    ax.set_xlabel("Période (jours)")
    ax.set_ylabel("Puissance")
    if SHOW_TITLES and title:
        ax.set_title(title, fontsize=12, fontweight="bold")
    ax.grid(True, which="both", alpha=0.3)
    ax.xaxis.set_minor_locator(LogLocator(subs="auto"))
    ax.xaxis.set_minor_formatter(NullFormatter())
    ax.legend(loc="upper right", frameon=True)


def prepare_series_for_key(data_by_exp, base_key):
    """Charge/clean/crop la série demandée pour chaque expérience."""
    pk, yk = f"{base_key}_periods", f"{base_key}_power"

    # Nettoyage
    cleaned = {}
    for key, data in data_by_exp.items():
        if pk in data and yk in data:
            p, y = clean_series(data[pk], data[yk])
            cleaned[key] = (p, normalize(y))

    # Recadrage à l'intersection commune
    if USE_COMMON_RANGE and cleaned:
        pmin, pmax = intersect_range_multi(cleaned)
        if pmin is not None and pmin < pmax:
            for k in list(cleaned.keys()):
                p, y = cleaned[k]
                cleaned[k] = crop_to_range(p, y, pmin, pmax)
    return cleaned


def compute_zoom_window(series_dict, ref_key="proxies", frac=0.20):
    """
    Détermine la fenêtre de zoom autour du pic principal de ref_key.
    """
    if ref_key not in series_dict:
        # sinon, prend n'importe laquelle avec données
        ref_key = next((k for k, (p, y) in series_dict.items() if len(p) > 0), None)
        if ref_key is None:
            return None

    p_ref, y_ref = series_dict[ref_key]
    if len(p_ref) == 0:
        return None
    peaks = find_top_peaks(p_ref, y_ref, k=1, exclude_edge_frac=PEAK_EXCLUDE_EDGE)
    if not peaks:
        return None
    p_peak = p_ref[peaks[0]]
    return p_peak * (1 - frac), p_peak * (1 + frac), p_peak


def crop_series_dict(series_dict, pmin, pmax):
    out = {}
    for k, (p, y) in series_dict.items():
        out[k] = crop_to_range(p, y, pmin, pmax)
    return out


# =========
# Main
# =========
def main():
    # Charger les 3 expériences (si dispo)
    data_by_exp = {}
    for key, path in DATA_PATHS.items():
        if path.exists():
            data_by_exp[key] = np.load(path, allow_pickle=True)
        else:
            print(f"[WARN] Fichier absent: {key} -> {path}")

    if not data_by_exp:
        print("[ERR] Aucun fichier .npz chargé.")
        return

    for base_key, label in SERIES:
        # Préparer les séries globales
        series_global = prepare_series_for_key(data_by_exp, base_key)

        # ---- Figure globale (3 expériences) ----
        fig, ax = plt.subplots(figsize=FIGSIZE_SINGLE, dpi=DPI)
        plot_many(ax, series_global, title=f"Periodogram – {label}")
        fig.tight_layout()
        out = OUTDIR / f"periodogram_{base_key}_baseline_vs_proxies_vs_restframe.png"
        fig.savefig(out)
        plt.close(fig)
        print(f"[OK] Sauvé: {out}")

        # ---- Zoom autour du pic (défini via 'proxies' si dispo) ----
        zoom = compute_zoom_window(series_global, ref_key="proxies", frac=0.20)
        if zoom is None:
            print(f"[WARN] Pas de fenêtre de zoom pour {base_key}")
            continue
        pmin, pmax, p_peak = zoom
        series_zoom = crop_series_dict(series_global, pmin, pmax)

        fig, ax = plt.subplots(figsize=(6.0, 4.2), dpi=DPI)
        plot_many(ax, series_zoom, title=None)
        # titre vide (comme tu avais fait)
        ax.set_title("", fontsize=12, fontweight="bold")
        fig.tight_layout()
        outz = (
            OUTDIR / f"periodogram_{base_key}_zoom_baseline_vs_proxies.png"
        )
        fig.savefig(outz)
        plt.close(fig)
        print(f"[OK] Sauvé: {outz}")


if __name__ == "__main__":
    main()


[OK] Sauvé: plots_compare_periodograms/periodogram_v_correct_baseline_vs_proxies_vs_restframe.png
[OK] Sauvé: plots_compare_periodograms/periodogram_v_correct_zoom_baseline_vs_proxies.png


In [22]:
dset = np.load(
    "/home/tliopis/Codes/exoplanets_llopis_mary_2025/data/npz_datasets/rvdatachallenge_ns1275_5000-5010.npz",
    allow_pickle=True,
)

In [ ]:
for key, value in dset.items():
    print(f"{key}: {value.shape}")

wavegrid: (1001,)
template: (1001,)
spectra: (1275, 1001)
time_values: (1275,)
v_true: (1275,)
metadata: ()
weights_fid: (1275, 1001)
sigma: (1001,)


In [2]:
# ============================================
# Periodograms (Lomb–Scargle) + viewer interactif
# ============================================
import numpy as np
import torch
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from src.dataset import generate_collate_fn
from scipy.signal import lombscargle
from src.modeling.train import load_experiment_checkpoint
from src.modeling.predict import (
    detrend_vencode_with_latent
)
ckpt = load_experiment_checkpoint(
    "../experiments/fine_tuned/models/aestra_final.pth",
    dataset_filepath="../data/npz_datasets/soapgpu_ns1275_5000-5010_snr2000_p100_k0p5_phi0.npz",
    device="cpu",
)

ckpt_dset = ckpt["dataset"]
ckpt_model = ckpt["model"]

# 1) Recréer un DataLoader SANS mélange (ordre temporel)
collate_fn = generate_collate_fn(ckpt_dset)
dataloader_ts = DataLoader(
    ckpt_dset, batch_size=64, shuffle=False, collate_fn=collate_fn
)

ckpt_model.eval()
ckpt_model.to("cpu")

S_list, vencode_list, yact_norm_list = [], [], []

with torch.no_grad():
    for batch in dataloader_ts:
        yobs = batch[0].to("cpu")  # [B, P]
        # même prétraitement que chez toi :
        y_centered = yobs - ckpt_model.b_rest

        vencode = ckpt_model.rvestimator(y_centered).detach().cpu().numpy().reshape(-1)
        yact, S = ckpt_model.spender(y_centered)
        S = S.detach().cpu().numpy()  # [B, 3]
        yact_norm = np.linalg.norm(yact.detach().cpu().numpy(), axis=1)  # [B]

        S_list.append(S)
        vencode_list.append(vencode)
        yact_norm_list.append(yact_norm)

S = np.concatenate(S_list, axis=0)  # [N, 3]
vencode = np.concatenate(vencode_list, axis=0)  # [N]
yact_norm = np.concatenate(yact_norm_list, axis=0)  # [N]

# 2) Récupérer les temps (on suppose alignés sur l'ordre du dataset)
t = np.asarray(ckpt_dset.time_values, dtype=float)  # ex: BJD
assert t.shape[0] == S.shape[0] == vencode.shape[0] == yact_norm.shape[0], (
    f"Longueurs incompatibles: t={t.shape}, S={S.shape}, v={vencode.shape}, ||y||={yact_norm.shape}"
)

/home/tliopis/Codes/exoplanets_llopis_mary_2025/venv/lib/python3.9/site-packages/numba/core/decorators.py:246: RuntimeWarning: nopython is set for njit and is ignored
  warnings.warn('nopython is set for njit and is ignored', RuntimeWarning)


[04:59:01] 📂 Loading experiment checkpoint: ../experiments/fine_tuned/models/aestra_final.pth         ]8;id=729649;file:///home/tliopis/Codes/exoplanets_llopis_mary_2025/src/modeling/train.py\train.py]8;;\:]8;id=238585;file:///home/tliopis/Codes/exoplanets_llopis_mary_2025/src/modeling/train.py#193\193]8;;\

In [4]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact
from astropy.timeseries import LombScargle

# detrending (si pas déjà fait)
v_correct, v0_avg, sigma_R = detrend_vencode_with_latent(vencode, S, knn=10)


def compute_ls_periodogram(t_days, y, min_period=2.0, max_period=None, oversample=5):
    """Retourne (periods, power) avec Astropy Lomb–Scargle."""
    t = np.asarray(t_days, float)
    y = np.asarray(y, float) - np.mean(y)

    Tspan = t.max() - t.min()
    if max_period is None:
        max_period = 0.9 * Tspan
    if t.size > 2:
        dt_med = np.median(np.diff(np.unique(t)))
        min_period = max(min_period, 2.0 * dt_med)

    # Grille de fréquences
    fmin = 1.0 / max_period
    fmax = 1.0 / min_period
    N = max(5000, int(oversample * max(1000, t.size)))
    freq = np.linspace(fmin, fmax, N)

    ls = LombScargle(t, y)
    power = ls.power(freq, normalization="psd")
    periods = 1.0 / freq
    return periods, power


%matplotlib qt
# Périodogramme
P, Pow = compute_ls_periodogram(t, v_correct, min_period=2.0)
plt.figure(figsize=(8, 3))
plt.plot(P, Pow)
plt.xscale("log")
plt.xlabel("Period [days] (log)")
plt.ylabel("LS power (Astropy)")
plt.axvline(27, ls="--", lw=1, c="r")
plt.grid(True, which="both")
plt.show()
